# Exploratory Data Analysis - Dropout Prediction

Looking at the synthetic dataset to understand distributions, correlations, and key patterns before building the ML pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# load the data
profiles = pd.read_csv('../data/student_profiles.csv')
records = pd.read_csv('../data/semester_records.csv')
ml_data = pd.read_csv('../data/ml_ready_dataset.csv')

print(f'Profiles: {profiles.shape}')
print(f'Semester records: {records.shape}')
print(f'ML dataset: {ml_data.shape}')

In [ ]:
profiles.head()

In [ ]:
ml_data.describe()

## Dropout rate analysis

In [ ]:
dropout_rate = ml_data['dropped_out'].mean()
print(f'Overall dropout rate: {dropout_rate:.1%}')

# by vulnerability group
print(f"\nOrphan dropout rate: {ml_data[ml_data['is_orphan']==True]['dropped_out'].mean():.1%}")
print(f"First-gen dropout rate: {ml_data[ml_data['is_first_gen']==True]['dropped_out'].mean():.1%}")
print(f"No guardian dropout rate: {ml_data[ml_data['has_guardian']==False]['dropped_out'].mean():.1%}")

In [ ]:
# dropout by income category
income_dropout = ml_data.groupby('income_category')['dropped_out'].mean().sort_values(ascending=False)
print(income_dropout)

income_dropout.plot(kind='bar', color=['#dc2626', '#ea580c', '#d97706', '#16a34a'])
plt.title('Dropout Rate by Income Category')
plt.ylabel('Dropout Rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# dropout by branch
branch_dropout = ml_data.groupby('branch')['dropped_out'].mean().sort_values(ascending=False)
print(branch_dropout)

branch_dropout.plot(kind='bar', color='steelblue')
plt.title('Dropout Rate by Branch')
plt.ylabel('Dropout Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Feature distributions

In [ ]:
# attendance distribution by dropout status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ml_data[ml_data['dropped_out']==False]['avg_attendance'].hist(ax=axes[0], bins=30, alpha=0.7, label='Retained', color='green')
ml_data[ml_data['dropped_out']==True]['avg_attendance'].hist(ax=axes[0], bins=30, alpha=0.7, label='Dropped', color='red')
axes[0].set_title('Avg Attendance Distribution')
axes[0].legend()

ml_data[ml_data['dropped_out']==False]['avg_ia_score'].hist(ax=axes[1], bins=30, alpha=0.7, label='Retained', color='green')
ml_data[ml_data['dropped_out']==True]['avg_ia_score'].hist(ax=axes[1], bins=30, alpha=0.7, label='Dropped', color='red')
axes[1].set_title('Avg IA Score Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap for key features
corr_cols = ['avg_attendance', 'avg_ia_score', 'total_subjects_failed', 
             'fee_defaults_count', 'avg_fee_delay', 'prev_academic_score',
             'avg_library_visits', 'dropped_out']

plt.figure(figsize=(10, 8))
sns.heatmap(ml_data[corr_cols].corr(), annot=True, cmap='RdYlBu_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# scatter: attendance vs IA score colored by dropout
plt.figure(figsize=(10, 6))
colors = ml_data['dropped_out'].map({True: 'red', False: 'green'})
plt.scatter(ml_data['avg_attendance'], ml_data['avg_ia_score'], c=colors, alpha=0.3, s=10)
plt.xlabel('Average Attendance %')
plt.ylabel('Average IA Score')
plt.title('Attendance vs IA Score (Red = Dropout)')
plt.tight_layout()
plt.show()

## Vulnerability analysis

In [ ]:
# cross-tab of vulnerability flags vs dropout
print('Orphan status:')
print(pd.crosstab(ml_data['is_orphan'], ml_data['dropped_out'], normalize='index'))

print('\nFirst-gen status:')
print(pd.crosstab(ml_data['is_first_gen'], ml_data['dropped_out'], normalize='index'))

print('\nGuardian status:')
print(pd.crosstab(ml_data['has_guardian'], ml_data['dropped_out'], normalize='index'))

In [ ]:
# semester-wise attendance trend for dropout vs retained
dropout_ids = ml_data[ml_data['dropped_out']==True]['student_id']
retained_ids = ml_data[ml_data['dropped_out']==False]['student_id']

dropout_trend = records[records['student_id'].isin(dropout_ids)].groupby('semester')['attendance_pct'].mean()
retained_trend = records[records['student_id'].isin(retained_ids)].groupby('semester')['attendance_pct'].mean()

plt.figure(figsize=(8, 5))
plt.plot(dropout_trend.index, dropout_trend.values, 'r-o', label='Dropout students')
plt.plot(retained_trend.index, retained_trend.values, 'g-o', label='Retained students')
plt.xlabel('Semester')
plt.ylabel('Avg Attendance %')
plt.title('Attendance Trend: Dropout vs Retained')
plt.legend()
plt.tight_layout()
plt.show()

## Key takeaways

1. Orphan students have ~80% dropout rate vs ~37% overall - biggest risk factor
2. First-gen learners dropout at ~53% - significantly higher than non-first-gen
3. BPL income students have highest dropout rate among income categories
4. Attendance and IA scores show clear separation between dropout and retained groups
5. Fee defaults are strongly correlated with dropout
6. Attendance degrades semester-over-semester for at-risk students - supports early warning approach